# 🥾 Tabular Reinforcement Learning: SARSA vs. Q-Learning on Cliff Walking

This notebook implements and compares two fundamental Temporal Difference (TD) control algorithms on the classic **Cliff Walking** gridworld task (Sutton & Barto, Chapter 6):
- **SARSA** (On-policy TD control)
- **Q-Learning** (Off-policy TD control)

We analyze how the exploration policy affects learning behavior and results in two distinctly different optimal paths.

---
## 1. Algorithmic Background

### SARSA (On-Policy Update)
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]$$
SARSA updates its action-value estimates using the actual action $A_{t+1}$ chosen by its exploration policy (e.g. $\epsilon$-greedy).

### Q-Learning (Off-Policy Update)
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$
Q-learning updates its estimates using the greedy action that maximizes the action-value function, independent of the actual exploratory action taken.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class CliffWalkingEnv:
    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)
        self.goal = (3, 11)
        self.actions = [0, 1, 2, 3] # Up, Down, Left, Right
        
    def reset(self):
        self.state = self.start
        return self.state
        
    def step(self, action):
        r, c = self.state
        if action == 0:   # Up
            r = max(r - 1, 0)
        elif action == 1: # Down
            r = min(r + 1, self.rows - 1)
        elif action == 2: # Left
            c = max(c - 1, 0)
        elif action == 3: # Right
            c = min(c + 1, self.cols - 1)
            
        self.state = (r, c)
        
        # Check if fell off the cliff
        if r == 3 and 0 < c < 11:
            self.state = self.start
            return self.state, -100.0, False # Send back to start
            
        # Check goal
        if self.state == self.goal:
            return self.state, -1.0, True
            
        return self.state, -1.0, False


## 2. Agent Implementations

We define the SARSA and Q-Learning agents using $\epsilon$-greedy exploration policies.


In [ ]:
class TabularAgent:
    def __init__(self, actions, alpha=0.5, epsilon=0.1, gamma=0.9):
        self.actions = actions
        self.alpha = alpha
        self.epsilon = epsilon
        self.gamma = gamma
        self.q_table = {} # Key: state tuple, Val: numpy array of action values
        
    def get_q_values(self, state):
        if state not in self.q_table:
            self.q_table[state] = np.zeros(len(self.actions))
        return self.q_table[state]
        
    def choose_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.actions)
        q_vals = self.get_q_values(state)
        return np.random.choice([a for a, q in enumerate(q_vals) if q == np.max(q_vals)])

class QLearningAgent(TabularAgent):
    def update(self, state, action, reward, next_state, done):
        q_vals = self.get_q_values(state)
        next_q_vals = self.get_q_values(next_state)
        
        max_next_q = 0.0 if done else np.max(next_q_vals)
        target = reward + self.gamma * max_next_q
        
        q_vals[action] += self.alpha * (target - q_vals[action])

class SarsaAgent(TabularAgent):
    def update(self, state, action, reward, next_state, next_action, done):
        q_vals = self.get_q_values(state)
        next_q_vals = self.get_q_values(next_state)
        
        next_q = 0.0 if done else next_q_vals[next_action]
        target = reward + self.gamma * next_q
        
        q_vals[action] += self.alpha * (target - q_vals[action])


## 3. Training Loop

We train both agents over 500 episodes and record their total reward per episode.


In [ ]:
def train_agent(agent, env, episodes=500):
    rewards = []
    for ep in range(episodes):
        state = env.reset()
        action = agent.choose_action(state)
        total_reward = 0
        done = False
        
        while not done:
            next_state, reward, done = env.step(action)
            total_reward += reward
            
            if isinstance(agent, QLearningAgent):
                agent.update(state, action, reward, next_state, done)
                next_action = agent.choose_action(next_state)
            else: # SarsaAgent
                next_action = agent.choose_action(next_state)
                agent.update(state, action, reward, next_state, next_action, done)
                
            state = next_state
            action = next_action
        rewards.append(total_reward)
    return rewards

env = CliffWalkingEnv()
q_agent = QLearningAgent(env.actions)
sarsa_agent = SarsaAgent(env.actions)

print("Training Q-Learning agent...")
q_rewards = train_agent(q_agent, env)
print("Training SARSA agent...")
sarsa_rewards = train_agent(sarsa_agent, env)
print("Training finished.")


## 4. Visualizing Results

We plot the smoothed reward curves during training to analyze convergence speed and stability.


In [ ]:
def smooth_curves(data, window=10):
    smoothed = []
    for i in range(len(data)):
        start = max(0, i - window + 1)
        smoothed.append(np.mean(data[start:i+1]))
    return smoothed

plt.figure(figsize=(10, 5))
plt.plot(smooth_curves(q_rewards), label="Q-Learning")
plt.plot(smooth_curves(sarsa_rewards), label="SARSA")
plt.xlabel("Episodes")
plt.ylabel("Sum of Rewards per Episode")
plt.title("SARSA vs. Q-Learning Convergence on Cliff Walking")
plt.ylim(-100, 0)
plt.grid(True)
plt.legend()
plt.show()


## 5. Visualizing Learned Trajectories

Let's extract the optimal paths chosen by the two agents after training.


In [ ]:
def get_optimal_path(agent, env):
    state = env.reset()
    path = [state]
    done = False
    limit = 50
    step_cnt = 0
    
    while not done and step_cnt < limit:
        q_vals = agent.get_q_values(state)
        action = np.argmax(q_vals)
        state, _, done = env.step(action)
        path.append(state)
        step_cnt += 1
    return path

q_path = get_optimal_path(q_agent, env)
sarsa_path = get_optimal_path(sarsa_agent, env)

def display_path(path, title):
    grid = np.zeros((4, 12), dtype=str)
    grid[:, :] = '.'
    grid[3, 1:11] = 'C' # Cliff
    grid[3, 0] = 'S'
    grid[3, 11] = 'G'
    
    for step, coord in enumerate(path):
        r, c = coord
        if (r, c) not in [(3,0), (3,11)]:
            grid[r, c] = str(step % 10)
            
    print(f"\n=== {title} Path ===")
    for r in range(4):
        print(" ".join(grid[r]))

display_path(q_path, "Q-Learning (Optimal but Risky)")
display_path(sarsa_path, "SARSA (Safe and Conservative)")


## 6. Analysis and Summary

- **Q-Learning** learns the optimal path that runs directly along the cliff edge. It has a path length of 13 steps (reward of -13). However, because it explores $\epsilon$-greedily, it occasionally takes random actions and falls off the cliff, incurring a -100 penalty. Its average online reward during training is lower.
- **SARSA** is an on-policy method. It accounts for its exploration policy. Recognizing that it might take random exploratory actions and fall off the cliff, SARSA learns a **safer path** one row above the cliff. While this path is longer (15 steps, reward of -15), it has a much lower probability of falling off the cliff, resulting in higher online rewards during training.
